# Telescope Data Classification - Part A

Classify telescope observations as gamma signal (`g`) or hadron background (`h`). This notebook implements the five Part A algorithms required by the capstone guidelines.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
data_path = Path.cwd().parent / "data" / "telescope_data.csv"
if not data_path.exists():
    data_path = Path("data/telescope_data.csv")
telescope = pd.read_csv(data_path).drop(columns=["Unnamed: 0"], errors="ignore")
print(f"Dataset shape: {telescope.shape}")
display(telescope.head())
display(telescope.dtypes.to_frame("dtype"))
display(telescope.isna().sum().to_frame("missing_values"))
print(f"Duplicate rows: {telescope.duplicated().sum()}")
display(telescope["class"].value_counts().rename_axis("class").to_frame("count"))

The target is moderately imbalanced, so the train/test split is stratified to preserve the class proportions.

In [ ]:
feature_columns = telescope.drop(columns="class").columns.tolist()
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, feature in zip(axes.ravel(), feature_columns):
    sns.histplot(data=telescope, x=feature, hue="class", bins=30, stat="density", common_norm=False, element="step", ax=ax)
    ax.set_title(feature)
plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 7))
sns.heatmap(telescope[feature_columns].corr(), cmap="coolwarm", center=0)
plt.title("Feature correlation heatmap")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=telescope, x="fLength", y="fWidth", hue="class", alpha=0.35, ax=axes[0])
axes[0].set_title("Length vs width")
sns.scatterplot(data=telescope, x="fSize", y="fConc", hue="class", alpha=0.35, ax=axes[1])
axes[1].set_title("Size vs concentration")
plt.tight_layout()
plt.show()

The feature distributions overlap, so combining measurements is preferable to relying on one feature. Standardisation is especially important for distance- and margin-based classifiers.

In [ ]:
model_data = telescope.copy()
model_data["length_width_ratio"] = model_data["fLength"] / model_data["fWidth"].replace(0, np.nan)
model_data["length_width_ratio"] = model_data["length_width_ratio"].replace([np.inf, -np.inf], np.nan)
X = model_data.drop(columns="class")
y = model_data["class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
numeric_preprocessor = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
results = []
predictions = {}

def evaluate_classifier(name, estimator):
    fitted = clone(estimator)
    fitted.fit(X_train, y_train)
    y_pred = fitted.predict(X_test)
    predictions[name] = y_pred
    results.append({"Model": name, "Accuracy": accuracy_score(y_test, y_pred), "Precision (weighted)": precision_score(y_test, y_pred, average="weighted", zero_division=0), "Recall (weighted)": recall_score(y_test, y_pred, average="weighted", zero_division=0), "F1 (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0)})
    print(f"{name} classification report")
    print(classification_report(y_test, y_pred, zero_division=0))
    display(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=["g", "h"]), index=["Actual g", "Actual h"], columns=["Predicted g", "Predicted h"]))
    return fitted

def show_results():
    return pd.DataFrame(results).sort_values("F1 (weighted)", ascending=False).reset_index(drop=True)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

## Logistic Regression

Logistic Regression is the linear baseline. Its coefficients describe the effect of standardised feature changes on class log-odds.

In [ ]:
logistic_model = Pipeline([("preprocessor", numeric_preprocessor), ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
logistic_fitted = evaluate_classifier("Logistic Regression", logistic_model)
show_results()

## K-Nearest Neighbours

KNN classifies an observation using the majority class among nearby observations. Scaling is required so high-magnitude features do not dominate the distance calculation; `k=5` provides a reproducible starting point.

In [ ]:
knn_model = Pipeline([("preprocessor", numeric_preprocessor), ("classifier", KNeighborsClassifier(n_neighbors=5, metric="minkowski"))])
knn_fitted = evaluate_classifier("K-Nearest Neighbours", knn_model)
show_results()

## Gaussian Naive Bayes

Gaussian Naive Bayes models each feature with a class-conditional Gaussian distribution and assumes conditional independence between features given the class. The assumption is strong, but the model is fast and provides a useful probabilistic baseline.

In [ ]:
naive_bayes_model = Pipeline([("preprocessor", numeric_preprocessor), ("classifier", GaussianNB())])
naive_bayes_fitted = evaluate_classifier("Gaussian Naive Bayes", naive_bayes_model)
show_results()